<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Frequency Domain — Theory</b></h1>
</div>

## Theoretical Foundations

This notebook documents the mathematical model, estimation methods, numerical considerations, diagnostics, and limitations used in the laboratory.
### Technical Context

The Fourier transform decomposes spatial image structure into frequency components whose magnitude and phase encode different aspects of the image.

### Core Frequency-Domain Model

For image $f(x,y)$, the 2-D DFT produces $F(u,v)$. Filtering multiplies the spectrum by a transfer function $H(u,v)$, then reconstructs with the inverse transform: $g=\mathcal{F}^{-1}\{HF\}$.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $f(x,y)$ | spatial-domain image |
| $F(u,v)$ | 2-D Fourier transform |
| $H(u,v)$ | frequency-domain filter |
| $G(u,v)$ | filtered spectrum |
| $D(u,v)$ | radial frequency distance |
| $D_0$ | cutoff frequency |

### Analytical Scope

Interpret spectra, distinguish magnitude/phase, design common frequency filters, explain ringing, remove periodic noise with notches, correct slowly varying illumination, and validate reconstruction/filtering.


## 1. Spatial-Frequency Characterization

A sinusoidal brightness pattern can be written as:

$$
g(x)=A\sin(2\pi f x+\phi)
$$

where:

- $A$ = amplitude;
- $f$ = spatial frequency;
- $\phi$ = phase.

Low spatial frequency means intensity changes slowly across space.  
High spatial frequency means intensity changes rapidly.

**Important:** high frequency does not mean high brightness.


## 2. 1-D DFT Validation with Synthetic Sinusoids

The DFT of a 1-D signal is:

$$
X[k]
=
\sum_{n=0}^{N-1}
x[n]e^{-j2\pi kn/N}
$$

Inverse:

$$
x[n]
=
\frac{1}{N}
\sum_{k=0}^{N-1}
X[k]e^{j2\pi kn/N}
$$

Euler's identity:

$$
e^{j\theta}
=
\cos(\theta)+j\sin(\theta)
$$

For $X=a+jb$:

$$
|X|=\sqrt{a^2+b^2}
$$

and

$$
\phi=\operatorname{atan2}(b,a)
$$

Magnitude = frequency strength.  
Phase = spatial alignment.


## 3. The 2-D Fourier Transform for Images

For image $f(x,y)$:

$$
F(u,v)
=
\sum_{x=0}^{M-1}
\sum_{y=0}^{N-1}
f(x,y)
e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

The FFT computes the DFT efficiently.

`fftshift` moves the zero-frequency component to the center:

- center → low frequencies;
- farther from center → high frequencies.

Before inverse FFT, undo the shift with `ifftshift`.


## 4. 2-D Spectrum Interpretation

A useful orientation rule:

> Spatial stripes produce spectral energy perpendicular to the stripe direction.

Let's prove it visually.


## 5. Inverse FFT and Reconstruction

The Fourier transform is reversible if we keep all coefficients.


## 6. Magnitude–Phase Analysis

Every Fourier coefficient can be written as:

$$
F(u,v)=|F(u,v)|e^{j\phi(u,v)}
$$

Magnitude tells us how strong a frequency is.  
Phase strongly controls spatial organization.


## 7. Frequency-Domain Filtering

Let $F$ be the image spectrum and $H$ the filter:

$$
G(u,v)=H(u,v)F(u,v)
$$

Then:

$$
g(x,y)=\mathcal{F}^{-1}\{G(u,v)\}
$$

Workflow:

1. FFT;
2. center with `fftshift`;
3. construct $H$;
4. multiply $H\cdot F$;
5. undo shift;
6. IFFT;
7. keep the real component.


## 8. Frequency Distance Grid

For circular filters:

$$
D(u,v)=
\sqrt{(u-u_0)^2+(v-v_0)^2}
$$


## 9. Ideal, Gaussian, and Butterworth Low-Pass Filters

### Ideal LPF

$$
H(u,v)=
\begin{cases}
1,&D(u,v)\le D_0\\
0,&D(u,v)>D_0
\end{cases}
$$

### Gaussian LPF

$$
H(u,v)
=
\exp\left(
-\frac{D(u,v)^2}{2D_0^2}
\right)
$$

### Butterworth LPF

$$
H(u,v)
=
\frac{1}
{1+\left(\frac{D(u,v)}{D_0}\right)^{2n}}
$$

Butterworth order $n$ controls transition steepness.


## 10. Ringing and the Gibbs Phenomenon

A hard spectral cutoff corresponds to an oscillatory spatial response:

> abrupt spectral boundary → spatial oscillations → halos near edges


## 11. High-Pass Filtering

For a normalized LPF:

$$
H_{HP}=1-H_{LP}
$$

High frequencies contain edges and fine detail, but can also contain noise.


## 12. High-Boost Sharpening

A pure high-pass result mainly contains detail.

For sharpening:

$$
g(x,y)=f(x,y)+k f_{HP}(x,y)
$$


## 13. Convolution Theorem

$$
f*h
\quad\Longleftrightarrow\quad
F\cdot H
$$

Spatial convolution corresponds to multiplication in the frequency domain.

### Circular vs Linear Convolution

A DFT assumes periodic extension.

Therefore direct FFT multiplication naturally performs **circular convolution**.  
For ordinary linear convolution, appropriate zero-padding is generally required.


## 14. Band-Pass and Band-Reject Filters

Band-pass keeps:

$$
D_1\le D(u,v)\le D_2
$$

Band-reject removes that interval.


## 15. Periodic Interference Analysis

Periodic interference is one of the strongest reasons to use the frequency domain.

Repeated interference often becomes isolated off-center peaks in the spectrum.


## 16. Spectral Peak Detection

The following detector is intentionally simple:

1. remove the central low-frequency area;
2. rank remaining coefficients;
3. keep strong points separated by a minimum distance.


## 17. Notch-Reject Filtering

A notch-reject filter suppresses a small neighborhood around selected unwanted frequencies.

Real images have conjugate-symmetric spectra, so corresponding symmetric frequencies must also be considered.


## 18. Moiré Removal

Moiré is a repeated interference pattern. It can often be easier to isolate in the Fourier domain than in the spatial domain.


## 19. Low-Frequency Illumination Correction

A simple multiplicative model is:

$$
I(x,y)\approx R(x,y)L(x,y)
$$

where:

- $R$ = reflectance / useful structure;
- $L$ = slowly varying illumination.

Because illumination varies slowly, it is dominated by low frequencies.


## 20. Cutoff Sensitivity

For a low-pass filter:

- smaller cutoff → stronger smoothing;
- larger cutoff → more detail preserved.


## 21. Quantitative Checks

MSE:

$$
\mathrm{MSE}
=
\frac{1}{MN}
\sum_{x,y}
[f(x,y)-g(x,y)]^2
$$

PSNR:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

PSNR measures numerical fidelity to a reference. It is not a universal perceptual-quality metric.


## 22. Validation Checks

This section develops the frequency-domain theory for validation checks.


## 23. Failure Modes and Diagnostic Signatures

Frequency-domain errors are often identifiable because they leave characteristic signatures in either the spectrum or the reconstructed image.

### Dynamic-Range Masking

The DC component and dominant low-frequency energy can be orders of magnitude larger than weak spectral peaks. A direct magnitude display can therefore appear almost black away from the origin even when meaningful periodic components are present. Logarithmic display compresses that dynamic range for diagnosis while leaving the actual complex spectrum unchanged for filtering.

### Shift Misalignment

A centered spectrum requires a transfer function designed around the centered frequency origin. Applying a mask in the wrong coordinate convention suppresses unintended frequencies even though the array dimensions remain valid. The error is therefore structural rather than syntactic.

### Abrupt Spectral Transitions

A hard cutoff produces a spatial response with oscillatory sidelobes. Near strong edges, those sidelobes appear as overshoot and undershoot. Ringing should therefore be diagnosed from edge profiles or overshoot measurements, not only from global image metrics.

### Sharpening and Noise

High-frequency enhancement does not distinguish useful fine structure from high-frequency acquisition noise. A sharpening configuration that increases edge contrast can simultaneously increase noise variance.

### Localized Spectral Suppression

A bright off-center spectral peak is not automatically interference. Repetitive texture can generate valid peaks. Notch placement must therefore be supported by spatial periodicity, conjugate symmetry, and before/after evidence.

### Circular-Convolution Artifacts

DFT-domain multiplication without adequate padding corresponds to periodic boundary assumptions. When the intended operation is ordinary linear convolution, insufficient padding produces wrap-around contamination near image boundaries.

### Diagnostic Principle

A valid frequency-domain result should be supported by multiple forms of evidence: spectral structure, spatial reconstruction, numerical checks, and consistency with the assumed image-formation or degradation mechanism.


## 24. Parameter Sensitivity and Controlled Experiments

Parameter selection is an experimental design problem. A meaningful sensitivity study varies one control while holding the remaining processing chain fixed.

### Cutoff Sensitivity

For low-pass filtering, decreasing the cutoff removes progressively more high-frequency structure. This generally increases smoothing and deviation from the original image. Increasing the cutoff preserves more detail but reduces the strength of the filtering intervention.

### Butterworth Order

The order controls how rapidly the transfer function changes around the cutoff. Low orders produce gradual transitions; higher orders approach a sharper boundary and therefore increase the possibility of ringing around strong spatial discontinuities.

### High-Boost Gain

The gain determines how strongly the extracted high-frequency component is added back to the image. Increasing it can improve local edge contrast while also increasing clipping and noise sensitivity.

### Notch Radius

A small notch may leave part of a periodic component untouched. An excessively large notch can remove neighboring frequencies that belong to useful image structure. Radius selection is therefore a localization trade-off.

### Controlled-Experiment Rule

For each sweep:

1. keep the input fixed;
2. keep non-target parameters fixed;
3. vary only the parameter under study;
4. record a quantitative response;
5. inspect the corresponding spatial or spectral output;
6. select a setting only after the trend is understood.

This makes the retained configuration reproducible and defensible.


## 25. Method Selection and Technical Discussion

Filter choice depends on the processing objective and on the failure modes that matter for that objective.

### Low-Pass Family

- **Ideal:** maximally sharp frequency separation, but the abrupt boundary makes ringing likely near strong edges.
- **Gaussian:** smooth transition and low ringing tendency, useful when artifact suppression and spatial smoothness are priorities.
- **Butterworth:** intermediate behavior with an explicit order parameter that controls transition steepness.

No family is universally best. The relevant comparison is between frequency selectivity, spatial artifacts, and preservation of task-relevant structure.

### Detail Enhancement

A high-pass image isolates rapidly varying content. High-boost sharpening reinjects that content into the original image, which is usually more useful for visualization than displaying the high-pass component alone. Both methods require noise checks.

### Periodic Interference

Notch rejection is appropriate when unwanted periodic components form localized spectral peaks that can be separated from useful image content. The notch position and radius must be validated against the corresponding spatial artifact.

### Illumination Variation

Slowly varying illumination is concentrated near low spatial frequencies. Frequency-domain estimation can therefore separate broad illumination trends from faster reflectance structure, but the correction model must remain consistent with the assumed image-formation process.

### Evidence-Based Selection

Method selection should combine quantitative change, edge preservation, ringing, clipping or noise behavior, and spectral localization. These criteria may favor different methods for different tasks; the final choice should state which objective is being optimized.


## 26. Integrated Frequency-Domain Workflow

A complete frequency-domain workflow links diagnosis, filter design, reconstruction, and validation rather than treating them as independent operations.

~~~text
Input image
    ↓
Input validation and spatial inspection
    ↓
2-D transform and centering
    ↓
Spectrum diagnosis
    ↓
Task-specific filter selection
    ↓
Parameter selection from controlled evidence
    ↓
Complex-spectrum filtering
    ↓
Inverse centering and reconstruction
    ↓
Spatial + spectral + numerical validation
    ↓
Configuration report and reproducible output
~~~

The workflow separates three responsibilities:

1. **Diagnosis:** determine what spectral structure corresponds to the processing objective.
2. **Intervention:** construct the smallest justified frequency-domain modification.
3. **Validation:** verify that the intended structure changed while unacceptable artifacts were not introduced.

The same execution pattern supports low-pass smoothing, high-frequency enhancement, band isolation, notch-based periodic-noise suppression, moiré reduction, and low-frequency illumination estimation.


## Technical Synthesis

Frequency-domain processing is expressed by the transfer-function formulation

$$
G(u,v)=H(u,v)F(u,v),
$$

with reconstruction through the inverse Fourier transform. The complete analysis chain is

$$
\boxed{
\text{image}
\rightarrow
\mathcal{F}
\rightarrow
\text{spectrum interpretation}
\rightarrow
H(u,v)
\rightarrow
\mathcal{F}^{-1}
\rightarrow
\text{spatial result}
\rightarrow
\text{diagnostics}
}
$$

Filter family, cutoff, order, spectral localization, phase preservation, conjugate symmetry, ringing, and periodic-noise signatures determine whether a frequency-domain intervention is justified.

## Scope and Limitations

### Included

2-D DFT/IDFT, spectrum reading, magnitude/phase, low/high/band filters, ringing, convolution theorem, periodic-noise detection/removal, moiré, shading, sensitivity, and validation.

### Not included

Wavelets and advanced multiresolution methods.


## References

- Gonzalez & Woods, *Digital Image Processing*
- Course slides, Image Processing, Centrale Nantes, 2025–2026
- NumPy FFT documentation: https://numpy.org/doc/stable/reference/routines.fft.html
